# GigaGraph 3.2B: SOTA-Grade LLM Pre-Training
### Distributed Cold Start | Remote Source (GitHub Sync)

**Hardware Target:** Kaggle Dual T4 (2x16GB)
**Architecture:** GigaGraph v8.1.1 (Synced from GitHub)
**Source Repo:** https://github.com/ey3lock3r/gnn-llm.git

In [1]:
# 1. Environment & Auth
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os, sys
os.environ['PATH'] = f"{os.environ['HOME']}/.local/bin:{os.environ['PATH']}"

from dotenv import load_dotenv
from kaggle_secrets import UserSecretsClient
load_dotenv()

try:
    sc = UserSecretsClient()
    hf_token = sc.get_secret("HF_TOKEN")
    wandb_key = sc.get_secret("WANDB_API_KEY")
except:
    hf_token, wandb_key = os.getenv('HF_TOKEN'), os.getenv('WANDB_API_KEY')

import torch, wandb
from huggingface_hub import login
if hf_token: login(token=hf_token)
if wandb_key: wandb.login(key=wandb_key)


In [2]:
# 2. Remote Synchronize (GitHub -> Kaggle)
REPO_URL = "https://github.com/ey3lock3r/gnn-llm.git"

print(f"Synchronizing source from {REPO_URL}...")
!git init .
!git remote add origin {REPO_URL} || git remote set-url origin {REPO_URL}
!git fetch origin
!git checkout -f master

# Install dependencies from the synchronized lockfile
!uv sync

# Ensure imports work from the current directory
if os.getcwd() not in sys.path: sys.path.append(os.getcwd())

from aptp_gnn import GigaGraph_3B
from data_pipeline import GigaDataPipeline
from tqdm import tqdm

In [3]:
# 3. GigaGraph 3.2B Training Loop
VOCAB_SIZE = 128256
D_MODEL = 3072
DEPTH = 32
LEARNING_RATE = 1e-4

model = GigaGraph_3B(vocab_size=VOCAB_SIZE, depth=DEPTH, d_model=D_MODEL)
pipeline = GigaDataPipeline()
loader = pipeline.get_dataloader(batch_size=2, seq_len=1024)

wandb.init(project="gigagraph-3b-cold-start")

print("🚀 GigaGraph 3B Launching (Remote Mode)...")
for i, batch in enumerate(tqdm(loader)):
    x = batch.to("cuda:0")
    y = torch.roll(x, -1, dims=1)
    loss = model.train_step(x, y, lr=LEARNING_RATE)
    if i % 10 == 0:
        wandb.log({"loss": loss.item()})
wandb.finish()